# Bayesian Possession Value Model
### Estimating the probability that a soccer possession results in a goal using Bayesian logistic regression

**Author:** Kiran Shah — MS Analytics, Georgia Tech | Former DC United Analytics  
**Full writeup:** [Medium Article](YOUR_MEDIUM_LINK_HERE)  
**Data:** [StatsBomb Open Data](https://github.com/statsbomb/open-data)

---

This notebook builds a Bayesian logistic regression model to estimate the value of individual actions based on possession results, using event-level data from the StatsBomb open data
repository. Rather than evaluating individual actions in isolation (as traditional metrics like
xG do), this framework evaluates entire possession sequences — capturing the buildup, not
just the finish.

Three prior specifications are tested to assess the sensitivity of results to prior assumptions:
- **Baseline:** Normal(0, 1.5) — semi-informative
- **Uninformative:** Normal(0, 10)
- **Strict:** Normal(0, 0.5) — regularizing

Features that hold up across all three specifications represent the most trustworthy signal in the data.

In [ ]:
# Data was accessed via Google Drive during development.
# To run locally, download the StatsBomb open data from:
# https://github.com/statsbomb/open-data
# and update the event_dir path below accordingly.

import json
import glob
import os
# (remaining imports follow in Cell 2)

## 1. Data Loading & Feature Engineering

StatsBomb event data represents every on-ball action in a match as a structured JSON object,
including the event type, location, and various type-specific attributes. Events are grouped
into possessions — continuous sequences of actions by the same team.

The target variable is binary: did a given possession result in a goal? Features are engineered
at the event level and capture the type, location, and timing of each action within its possession.
Temporal interaction terms (e.g. `attacking_third_timed`) are constructed by multiplying each
binary feature by the event's relative position in the possession sequence (0 = first event,
1 = last event), allowing the model to capture how the predictive value of each action type
scales with timing.

In [ ]:
# Data was accessed via Google Drive during development.
# To run locally, download the StatsBomb open data from:
# https://github.com/statsbomb/open-data
# and update event_dir below to point to your local events folder.

import json
import os
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from collections import defaultdict
from math import atan2, degrees, sqrt
from sklearn.preprocessing import StandardScaler

# UPDATE THIS PATH to your local StatsBomb data directory
event_dir = "path/to/statsbomb/open-data/data/events"

match_ids = list(range(19734, 19753))

# Helper Functions
def get_angle_and_distance(x1, y1, x2, y2):
    if None in (x1, y1, x2, y2):
        return None, None
    dx, dy = x2 - x1, y2 - y1
    angle = degrees(atan2(dy, dx))
    distance = sqrt(dx**2 + dy**2)
    return angle, distance

# Load Events
all_events = []
for match_id in match_ids:
    path = os.path.join(event_dir, f"{match_id}.json")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for e in data:
        e["match_id"] = match_id
        all_events.append(e)

In [ ]:
# Process events and feature engineering
possessions = defaultdict(list)
for e in all_events:
    if "possession" in e:
        key = (e["match_id"], e["possession"])
        possessions[key].append(e)

rows = []
for key, events in possessions.items():
    match_id, possession_id = key
    success = any(
        e["type"]["name"] == "Shot" and e.get("shot", {}).get("outcome", {}).get("name") == "Goal"
        for e in events
        )

    for e in events:
        loc = e.get("location", [None, None])
        x1, y1 = loc if loc else (None, None)
        end_loc = e.get(e["type"]["name"].lower(), {}).get("end_location", [None, None])
        x2, y2 = end_loc[:2] if end_loc else (None, None)
        angle, distance = get_angle_and_distance(x1, y1, x2, y2)
        in_attacking_third = int(x1 is not None and x1 >= 66)
        under_pressure = int(e.get("under_pressure", False))

        event_type = e["type"]["name"]
        dribble = int(event_type == "Dribble")
        carry = int(event_type == "Carry")
        if event_type in {"Foul Committed", "Interception"}:
            continue

        # Initialize pass traits
        pass_traits = {
            "progressive_pass": 0,
            "switch": 0,
            "cross": 0,
            "through_ball": 0,
            "cutback": 0
            }

        if event_type == "Pass":
            pass_data = e.get("pass", {})

            # Through ball
            if pass_data.get("type", {}).get("name") == "Through Ball":
                pass_traits["through_ball"] = 1

            # Cutback
            if pass_data.get("cut_back", False):
                pass_traits["cutback"] = 1

            # Progressive
            if x1 is not None and x2 is not None and x2 - x1 > 10:
                pass_traits["progressive_pass"] = 1

            # Switch
            if distance and abs(angle) > 75 and distance > 30:
                pass_traits["switch"] = 1

            # Cross
            if x1 and y1 and x1 > 70 and (y1 < 20 or y1 > 80) and angle < -20:
                pass_traits["cross"] = 1

        row = {
            "match_id": match_id,
            "team": e["team"]["name"],
            "possession": possession_id,
            "event_type": event_type,
            "dribble": dribble,
            "carry": carry,
            "angle": angle,
            "distance": distance,
            "under_pressure": under_pressure,
            "attacking_third": in_attacking_third,
            "success": success,
            **pass_traits
        }
        rows.append(row)

df = pd.DataFrame(rows)

# Add event index within each possession
df["event_index"] = df.groupby(["match_id", "possession"]).cumcount()

# Total number of events per possession
df["total_events"] = df.groupby(["match_id", "possession"])["event_type"].transform("count")

# Relative position of the event in the possession (mathematically guaranteed to be between 0 and 1)
df["event_position"] = df["event_index"] / (df["total_events"] - 1).replace(0, 1)

df = df.dropna(subset=["angle", "distance"])

# One-hot encode event type
df = pd.get_dummies(df, columns=["event_type"], drop_first=True)
df = df.drop(columns=["event_type_Pass"], errors="ignore")
df = df.rename(columns={'event_type_Shot': 'shot'})


base_feature_cols = [
    "under_pressure", "attacking_third",
    "progressive_pass", "switch", "cross",
    "cutback", "through_ball", "dribble", "carry", "shot"
]

for col in base_feature_cols:
    interaction_col = f"{col}_timed"
    df[interaction_col] = df[col] * df["event_position"]

# Standardize numerical predictors
scaler = StandardScaler()
df[["angle", "distance", "event_position"]] = scaler.fit_transform(
    df[["angle", "distance", "event_position"]]
)


# Final predictors and target
drop_cols = ["success", "match_id", "possession", "team", "event_index", "total_events"]
predictors = [col for col in df.columns if col not in drop_cols]

X = df[predictors].astype(np.float32).values
y = df["success"].astype(np.int32).values

## 2. Model Fitting

### Model 1 — Baseline Prior: Normal(0, 1.5)

The baseline model uses a semi-informative prior on the coefficients. A standard deviation
of 1.5 on the log-odds scale is wide enough to allow the data to speak but provides mild
regularization against extreme estimates — a reasonable default for a dataset of this size.

The intercept uses a tighter Normal(0, 1) prior, consistent with the expected low base rate
of goal-scoring possessions (~1% empirically).

In [ ]:
# Create the Bayesian model - medium informative (sigma = 1.5 on betas).
with pm.Model() as model:
    X_data = pm.Data("X_data", X)
    y_data = pm.Data("y_data", y)

    intercept = pm.Normal("intercept", 0, sigma=1)
    betas = pm.Normal("betas", 0, sigma=1.5, shape=X.shape[1])

    # Logistic likelihood
    logit_p = intercept + pm.math.dot(X_data, betas)
    success = pm.Bernoulli("success", logit_p=logit_p, observed=y_data)

    # Sample from posterior
    trace = pm.sample(1000, tune=200, target_accept=0.95)

In [ ]:
# Summarize posterior
summary = az.summary(trace, var_names=["intercept", "betas"], hdi_prob=0.95)
summary.index = ["intercept"] + list(predictors)
print(summary)

### Model 2 — Uninformative Prior: Normal(0, 10)

This specification places minimal constraint on the coefficients, allowing the posterior to
be driven almost entirely by the data. The purpose is to test whether the baseline results
are an artifact of prior choice — if key features remain significant under an uninformative
prior, that strengthens the case for their importance.

As expected, sparse features like `cutback` and `through_ball` produce more extreme and
unstable estimates under this specification, illustrating the risk of uninformative priors
in low-signal sports data.

In [ ]:
# Recreate the model - uninformative priors (sigma = 10 on betas)
with pm.Model() as model_uninformative:
    X_data = pm.Data("X_data", X)
    y_data = pm.Data("y_data", y)

    intercept = pm.Normal("intercept", 0, sigma=1)
    betas = pm.Normal("betas", 0, sigma=10, shape=X.shape[1])

    # Logistic likelihood
    logit_p = intercept + pm.math.dot(X_data, betas)
    success = pm.Bernoulli("success", logit_p=logit_p, observed=y_data)

    # Sample from posterior
    trace_uninformative = pm.sample(1000, tune=200, target_accept=0.95)


In [ ]:
# Analyze the uninformative prior model
summary_uninformative = az.summary(trace_uninformative, var_names=["intercept", "betas"], hdi_prob=0.95)
summary_uninformative.index = ["intercept"] + list(predictors)
print(summary_uninformative)

### Model 3 — Strict Prior: Normal(0, 0.5)

This specification applies stronger regularization, shrinking coefficients toward zero more
aggressively. Features that remain significant under this prior are robust to the most
conservative assumptions. The tradeoff is that genuine but modest effects may be suppressed.

Comparing results across all three specifications provides a fuller picture of which findings
are stable and which are sensitive to modeling assumptions.

In [ ]:
# Recreate the model - informative priors (sigma = 0.5 on betas)
with pm.Model() as model_informative:
    X_data = pm.Data("X_data", X)
    y_data = pm.Data("y_data", y)

    intercept = pm.Normal("intercept", 0, sigma=1)
    betas = pm.Normal("betas", 0, sigma=0.5, shape=X.shape[1])

    # Logistic likelihood
    logit_p = intercept + pm.math.dot(X_data, betas)
    success = pm.Bernoulli("success", logit_p=logit_p, observed=y_data)

    # Sample from posterior
    trace_informative = pm.sample(1000, tune=200, target_accept=0.95)

In [ ]:
# Analyze the informative prior model
summary_informative = az.summary(trace_informative, var_names=["intercept", "betas"], hdi_prob=0.95)
summary_informative.index = ["intercept"] + list(predictors)
print(summary_informative)

## 3. Posterior Diagnostics

Standard convergence checks for the baseline model. `az.plot_posterior` displays the full
posterior distribution for each parameter. `az.plot_forest` provides a compact comparison
of posterior means and credible intervals across all coefficients.

R-hat values close to 1.0 and well-mixed trace plots indicate that the sampler converged
successfully.

In [ ]:
az.plot_posterior(trace)

In [ ]:
az.plot_forest(trace, combined=True)

## 4. Visualization

### Coefficient Plot — Baseline Model

Posterior means and 95% highest density intervals (HDI) for all coefficients in the baseline
model, sorted by effect size. Features whose HDI excludes zero are highlighted in green and
represent the most reliable predictors of possession success. Features whose HDI overlaps
zero are shown in grey — their direction of effect is uncertain given the data.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Drop intercept, just plot betas
df = summary.drop("intercept")

# Sort by mean for readability
df = df.sort_values("mean")

# Color: highlight bars whose HDI excludes zero
colors = ["#2D6A4F" if (row["hdi_2.5%"] > 0 or row["hdi_97.5%"] < 0) else "#AAAAAA"
          for _, row in df.iterrows()]

fig, ax = plt.subplots(figsize=(9, 10))
fig.patch.set_facecolor("#F5F2EC")
ax.set_facecolor("#F5F2EC")

y_pos = np.arange(len(df))

# HDI bars
ax.barh(y_pos, df["hdi_97.5%"] - df["hdi_2.5%"],
        left=df["hdi_2.5%"], height=0.5,
        color=colors, alpha=0.3)

# Mean dots
ax.scatter(df["mean"], y_pos, color=colors, zorder=5, s=40)

# Zero line
ax.axvline(0, color="#1A1A1A", linewidth=0.8, linestyle="--", alpha=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(df.index, fontsize=9, fontfamily="monospace")
ax.set_xlabel("Posterior Coefficient (log-odds)", fontsize=9)
ax.set_title("Posterior Coefficients — Baseline Model\n95% HDI, excluding intercept",
             fontsize=12, fontfamily="serif", fontstyle="italic", pad=12)

sig_patch = mpatches.Patch(color="#2D6A4F", alpha=0.6, label="95% HDI excludes zero")
ins_patch = mpatches.Patch(color="#AAAAAA", alpha=0.6, label="HDI overlaps zero")
ax.legend(handles=[sig_patch, ins_patch], fontsize=8, framealpha=0)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.savefig("coefficient_plot.png", dpi=180, bbox_inches="tight", facecolor="#F5F2EC")

### Prior Comparison — Selected Predictors

Forest plot comparing posterior estimates across all three prior specifications for a subset
of the most informative features. Filled dots indicate that the 95% HDI excludes zero
(significant); open dots indicate the HDI overlaps zero (uncertain).

Features that are significant across all three specifications — particularly `shot`,
`attacking_third_timed`, and `event_position` — represent the most robust findings of
the analysis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
# Only show the predictors that are interesting to compare - the ones that
# are significant in at least one model, or that shift meaningfully across priors.
# Adjust this list to your taste.
focus_features = [
    "shot", "attacking_third_timed", "attacking_third",
    "event_position", "switch", "progressive_pass",
    "carry", "cutback", "under_pressure_timed", "shot_timed"
]

models = {
    "Baseline\nNormal(0, 1.5)":      summary,
    "Uninformative\nNormal(0, 10)":   summary_uninformative,
    "Strict\nNormal(0, 0.5)":         summary_informative,
}

COLORS = {
    "Baseline\nNormal(0, 1.5)":     "#2D6A4F",
    "Uninformative\nNormal(0, 10)":  "#3730A3",
    "Strict\nNormal(0, 0.5)":        "#92400E",
}

# Layout
n_features = len(focus_features)
n_models = len(models)
group_h = 0.22        # height per model dot within a feature group
gap_h = 0.35          # extra gap between feature groups
y_positions = {}
y = 0
for feat in reversed(focus_features):   # reversed so top of plot = first in list
    y_positions[feat] = [y + i * group_h for i in range(n_models)]
    y += n_models * group_h + gap_h

fig_h = max(7, n_features * 1.1)
fig, ax = plt.subplots(figsize=(10, fig_h))
fig.patch.set_facecolor("#F5F2EC")
ax.set_facecolor("#F5F2EC")

# Plot
for feat in focus_features:
    for i, (model_name, df) in enumerate(models.items()):
        if feat not in df.index:
            continue
        row = df.loc[feat]
        yp = y_positions[feat][i]
        color = COLORS[model_name]

        # Check significance
        significant = row["hdi_2.5%"] > 0 or row["hdi_97.5%"] < 0
        alpha_hdi = 0.35 if significant else 0.15

        # HDI bar
        ax.barh(yp, row["hdi_97.5%"] - row["hdi_2.5%"],
                left=row["hdi_2.5%"], height=group_h * 0.55,
                color=color, alpha=alpha_hdi)

        # Mean dot — filled if significant, open if not
        ax.scatter(row["mean"], yp, color=color, zorder=5,
                   s=45, linewidths=1.2,
                   facecolors=color if significant else "#F5F2EC",
                   edgecolors=color)

# Y-axis labels (feature names, centered on each group)
ytick_pos = [np.mean(y_positions[f]) for f in focus_features]
ax.set_yticks(ytick_pos)
ax.set_yticklabels(focus_features, fontsize=9, fontfamily="monospace")

# Subtle horizontal band per feature for readability
for j, feat in enumerate(focus_features):
    ys = y_positions[feat]
    band_bottom = ys[0] - group_h * 0.4
    band_top    = ys[-1] + group_h * 0.4 + gap_h * 0.3
    if j % 2 == 0:
        ax.axhspan(band_bottom, band_top, color="#E8E4DC", alpha=0.35, zorder=0)

# Zero line
ax.axvline(0, color="#1A1A1A", linewidth=0.8, linestyle="--", alpha=0.4)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
           markeredgecolor=c, markersize=7, label=name.replace("\n", " "))
    for name, c in COLORS.items()
]
legend_elements += [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1A1A1A',
           markersize=7, label='Filled = 95% HDI excludes zero'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#F5F2EC',
           markeredgecolor='#1A1A1A', markersize=7, label='Open = HDI overlaps zero'),
]
ax.legend(handles=legend_elements, fontsize=8, framealpha=0,
          loc="lower right")

# Labels
ax.set_xlabel("Posterior Coefficient (log-odds)", fontsize=9)
ax.set_title("Coefficient Estimates Across Prior Specifications\n95% HDI — selected predictors",
             fontsize=12, fontfamily="serif", fontstyle="italic", pad=12)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.15)

plt.tight_layout()
plt.savefig("coefficient_comparison.png", dpi=180, bbox_inches="tight",
            facecolor="#F5F2EC")

## 5. Results Summary

- **Shot** is the strongest predictor of possession success across all model specifications
- **Attacking third entry timed** (`attacking_third_timed`) is consistently the second most
  important feature — the value of entering the attacking third scales up significantly the
  later in a possession it occurs
- **Event position** has a negative coefficient, suggesting that possessions which reach
  threatening actions earlier tend to be more dangerous on average
- The **baseline goal probability** implied by the intercept (~0.9%) matches the empirical
  rate in professional soccer without explicit calibration — a useful sanity check
- Results are stable across all three prior specifications for the key features,
  strengthening confidence in these findings
- Uninformative priors produced implausibly extreme estimates for sparse features like
  `cutback` and `through_ball`, illustrating the importance of prior choice in
  low-signal sports data

---

**Full writeup:** [Medium Article](YOUR_MEDIUM_LINK_HERE)  
**Repository:** [GitHub](YOUR_GITHUB_LINK_HERE)